<a href="https://colab.research.google.com/github/dubacchiega/metrobot/blob/main/MetroBot_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q groq ollama ipywidgets python-dotenv

In [ ]:
import os, json, re, unicodedata
from collections import deque
from itertools import product
PROVEDOR = "groq" # "groq" (nuvem), "ollama" (local) ou "offline" (sem LLM)
MODELO_GROQ = "openai/gpt-oss-20b" # confira em console.groq.com/docs/models
MODELO_OLLAMA = "llama3.2" # baixado com: ollama pull llama3.2
def obter_chave_groq():
  try:
    from google.colab import userdata
    return userdata.get("GROQ_API_KEY")
  except Exception:
    pass
  try:
      from dotenv import load_dotenv
      load_dotenv()
  except Exception:
    pass
  return os.environ.get("GROQ_API_KEY")
def chamar_llm(mensagens, modo_json=False):
  if PROVEDOR == "groq":
    from groq import Groq
    cliente = Groq(api_key=obter_chave_groq())
    extras = {"response_format": {"type": "json_object"}} if modo_json else {}
    resposta = cliente.chat.completions.create(
      model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)
    return resposta.choices[0].message.content
  elif PROVEDOR == "ollama":
    import ollama
    extras = {"format": "json"} if modo_json else {}
    resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
      options={"temperature": 0}, **extras)
    return resposta["message"]["content"]
  else:
    raise RuntimeError("Modo offline: nenhum LLM configurado.")

In [ ]:
resposta = chamar_llm([
  {"role": "user",
    "content": "Em uma frase curta, dê boas-vindas aos passageiros do metrô de São Paulo."}
])
print(resposta)

Bem-vindos ao metrô de São Paulo!


In [ ]:
from collections import deque

LINHAS = {
  "Linha 1-Azul": [
    "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
    "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
    "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
    "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
    "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
  ],
  "Linha 2-Verde": [
    "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
    "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
    "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
    "Vila Prudente",
  ],
  "Linha 3-Vermelha": [
    "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
    "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
    "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
    "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
    "Artur Alvim", "Corinthians-Itaquera",
  ],
}

CORES = {"Linha 1-Azul": "#1e88e5", "Linha 2-Verde": "#2e7d32",
"Linha 3-Vermelha": "#d32f2f"}

LOCAIS = {
    "Pinacoteca": "Luz",
    "Catedral da Sé": "Sé",
    "Shopping Metrô Santa Cruz": "Santa Cruz",
    "MASP": "Trianon-Masp",
    "Hospital das Clínicas": "Clínicas",
    "Museu do Ipiranga": "Alto do Ipiranga",
    "Theatro Municipal": "Anhangabaú",
    "Neo Química Arena": "Corinthians-Itaquera",
    "Memorial da América Latina": "Palmeiras-Barra Funda"
}

def construir_grafo_multilinhas(linhas):
    grafo = {}
    linhas_do_trecho = {}
    for nome_linha, estacoes in linhas.items():
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]
            if a not in grafo: grafo[a] = []
            if b not in grafo: grafo[b] = []

            if b not in grafo[a]: grafo[a].append(b)
            if a not in grafo[b]: grafo[b].append(a)

            linhas_do_trecho.setdefault((a, b), set()).add(nome_linha)
            linhas_do_trecho.setdefault((b, a), set()).add(nome_linha)

    return grafo, linhas_do_trecho

GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)
TODAS_ESTACOES = list(GRAFO.keys())


In [ ]:
def contar_baldeacoes(caminho, linhas_do_trecho):
    if not caminho or len(caminho) < 2: return 0, []
    baldeacoes = []
    linhas_atuais = linhas_do_trecho[(caminho[0], caminho[1])]

    for i in range(1, len(caminho) - 1):
        u, v = caminho[i], caminho[i+1]
        linhas_prox = linhas_do_trecho[(u, v)]
        intersecao = linhas_atuais.intersection(linhas_prox)

        if intersecao:
            linhas_atuais = intersecao
        else:
            nova_linha = list(linhas_prox)[0]
            baldeacoes.append((u, nova_linha))
            linhas_atuais = linhas_prox

    return len(baldeacoes), baldeacoes

In [ ]:
def reconstruir_caminho(pai, destino):
    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = pai[atual]
    return list(reversed(caminho))

In [ ]:
def bfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas: return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita

def dfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas: return None, []
    visitados = set()
    ordem_visita = []
    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino: return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])
                if resultado: return resultado
        return None
    caminho = explorar(origem, [origem])
    return caminho, ordem_visita

In [ ]:
def fatos_base():
    fatos = set()
    for linha, estacoes in LINHAS.items():
        for e in estacoes:
            fatos.add(("pertence", e, linha))
    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar(fatos, predicado):
    return [f[1:] for f in fatos if f[0] == predicado]

def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local: novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local: novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"): return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar(fatos, papel):
            if e in inacessiveis: novos.add(("alerta", papel, e))
    return novos

def r_integracao(fatos):
    pertence_a = {}
    for (estacao, linha) in consultar(fatos, "pertence"):
        pertence_a.setdefault(estacao, set()).add(linha)
    return {("integracao", e) for e, linhas in pertence_a.items() if len(linhas) > 1}

def r_linha_paralisada(fatos):
    novos = set()
    paralisadas = {l for (l,) in consultar(fatos, "linha_paralisada")}
    for (estacao, linha) in consultar(fatos, "pertence"):
        if linha in paralisadas:
            novos.add(("bloqueada", estacao))
    return novos

In [ ]:
REGRAS = [
    ("R1 origem", "", r_origem),
    ("R2 destino", "", r_destino),
    ("R3 bloqueio", "", r_bloqueio),
    ("R4 acessibilidade","", r_acessibilidade),
    ("R5 alerta", "", r_alerta),
    ("R6 integracao", "∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2 → integracao(e))", r_integracao),
    ("R7 linha parada", "∀e ∀l (pertence(e,l) ∧ linha_paralisada(l) → bloqueada(e))", r_linha_paralisada)
]

def encadear_para_frente(fatos, regras):
    fatos = set(fatos)
    justificativas = {}
    while True:
        novos = set()
        for nome, _, regra in regras:
            for fato in regra(fatos) - fatos:
                novos.add(fato)
                justificativas[fato] = nome
        if not novos: return fatos, justificativas
        fatos |= novos

In [ ]:
def tabela_verdade_acessibilidade():
    print("Regra Proposicional: Precisa de Acessibilidade ∧ Elevador em Manutenção → Inacessível")
    print(" P (Acessibilidade) | Q (Manutenção) | P ∧ Q (Inacessível)")
    print("-" * 65)
    for p, q in product([True, False], repeat=2):
        print(f" {str(p):<18} | {str(q):<14} | {str(p and q)}")

In [ ]:
def planejar(pedido, fechadas=(), manutencao=(), algoritmo="BFS", linhas_paralisadas=()):
    fatos = fatos_base()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]

    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))
    if pedido.get("acessibilidade"): fatos.add(("precisa_acessibilidade",))

    for e in fechadas: fatos.add(("fechada", e))
    for e in manutencao: fatos.add(("elevador_em_manutencao", e))
    for l in linhas_paralisadas: fatos.add(("linha_paralisada", l))

    fatos, justificativas = encadear_para_frente(fatos, REGRAS)

    origem = consultar(fatos, "origem")[0][0]
    destino = consultar(fatos, "destino")[0][0]
    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
    alertas = consultar(fatos, "alerta")

    buscar = bfs if algoritmo == "BFS" else dfs
    caminho, visitados = buscar(GRAFO, origem, destino, bloqueadas)

    qtd_baldeacoes, baldeacoes = contar_baldeacoes(caminho, LINHAS_DO_TRECHO) if caminho else (0, [])

    return {
        "origem": origem, "destino": destino, "algoritmo": algoritmo,
        "caminho": caminho, "visitados": visitados, "bloqueadas": sorted(bloqueadas),
        "alertas": [f"{papel}: {e}" for papel, e in alertas],
        "paradas": len(caminho) - 1 if caminho else None,
        "baldeacoes": qtd_baldeacoes,
        "lista_baldeacoes": baldeacoes,
        "tempo_min": (len(caminho) - 1) * 2 + (qtd_baldeacoes * 5) if caminho else None,
        "regras_usadas": sorted(set(justificativas.values())),
    }

In [ ]:
def rodar_testes():
    # Caso 1
    r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Corinthians-Itaquera")}, algoritmo="BFS")
    assert r["paradas"] == 22 and r["baldeacoes"] == 1
    # Caso 2
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Jabaquara")}, algoritmo="BFS")
    assert r["paradas"] == 14 and r["baldeacoes"] == 1
    # Caso 3
    r = planejar({"origem": ("estacao", "Palmeiras-Barra Funda"), "destino": ("estacao", "Vila Prudente")}, algoritmo="BFS")
    assert r["paradas"] == 16 and r["baldeacoes"] == 2
    # Caso 4
    r = planejar({"origem": ("estacao", "Tucuruvi"), "destino": ("estacao", "Brás")}, fechadas=["Sé"])
    assert r["caminho"] is None
    # Caso 5
    r = planejar({"origem": ("estacao", "Vila Madalena"), "destino": ("estacao", "Jabaquara")}, fechadas=["Paraíso"])
    assert r["caminho"] is None
    # Caso 6
    r = planejar({"origem": ("estacao", "Vila Prudente"), "destino": ("estacao", "Jabaquara")}, fechadas=["Paraíso"])
    assert r["paradas"] == 13 and "Ana Rosa" in r["caminho"]

    # Customizados
    r = planejar({"origem": ("local", "MASP"), "destino": ("local", "Pinacoteca")})
    assert r["origem"] == "Trianon-Masp" and r["destino"] == "Luz"
    r = planejar({"origem": ("estacao", "Saúde"), "destino": ("estacao", "Sacomã")}, linhas_paralisadas=["Linha 2-Verde"])
    assert r["caminho"] is None



In [ ]:
rodar_testes()
print("Todos os testes passaram com sucesso.")
tabela_verdade_acessibilidade()

Todos os testes passaram com sucesso.
Regra Proposicional: Precisa de Acessibilidade ∧ Elevador em Manutenção → Inacessível
 P (Acessibilidade) | Q (Manutenção) | P ∧ Q (Inacessível)
-----------------------------------------------------------------
 True               | True           | True
 True               | False          | False
 False              | True           | False
 False              | False          | False


In [ ]:
PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP.
Sua única tarefa é transformar o pedido do passageiro em JSON.

Estações válidas: {estacoes}
Locais válidos: {locais}

Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>",
  "destino": "<nome exato de estação ou local, ou null>",
  "acessibilidade": <true ou false>}}

Regras:
- Use SOMENTE nomes das listas acima, escritos exatamente como aparecem.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas, mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
- Se não souber algum campo, use null. Nunca invente nomes."""

def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome(nome):
    if not nome: return None
    alvo = normalizar(nome).strip()
    for estacao in TODAS_ESTACOES:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)
    for local in LOCAIS:
        if normalizar(local) == alvo:
            return ("local", local)
    return None

def interpretar_offline(texto):
    texto_min = texto.lower()
    texto_sem = normalizar(texto)
    candidatos = [(n, "estacao") for n in TODAS_ESTACOES] + [(n, "local") for n in LOCAIS]
    candidatos.sort(key=lambda c: len(c[0]), reverse=True)

    ocupado = [False] * len(texto_min)
    encontrados = []

    for nome, tipo in candidatos:
        buscas = [(texto_min, nome.lower())]
        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append((texto_sem, normalizar(nome)))
        for base, padrao in buscas:
            for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", base):
                if not any(ocupado[m.start():m.end()]):
                    encontrados.append((m.start(), nome))
                    for i in range(m.start(), m.end()):
                        ocupado[i] = True
    encontrados.sort()
    palavras_acess = ["cadeira de rodas", "acessibilidade", "mobilidade", "muleta", "carrinho de bebe", "elevador"]
    return {
        "origem": encontrados[0][1] if len(encontrados) > 0 else None,
        "destino": encontrados[1][1] if len(encontrados) > 1 else None,
        "acessibilidade": any(p in texto_sem for p in palavras_acess),
    }

def interpretar_pedido(texto):
    if PROVEDOR == "offline":
        bruto = interpretar_offline(texto)
        fonte = "offline (RegEx/Fuzzy)"
    else:
        sistema = PROMPT_INTERPRETE.format(estacoes=", ".join(TODAS_ESTACOES), locais=", ".join(LOCAIS))
        try:
            resposta = chamar_llm([{"role": "system", "content": sistema},
                                    {"role": "user", "content": texto}], modo_json=True)
            bruto = json.loads(resposta)
            fonte = f"LLM ({PROVEDOR})"
        except Exception as erro:
            bruto = interpretar_offline(texto)
            fonte = f"offline (Fallback: {erro})"

    origem = resolver_nome(bruto.get("origem"))
    destino = resolver_nome(bruto.get("destino"))

    if origem is None or destino is None:
        return None, f"⚠️ Não foi possível identificar origem/destino claramente via {fonte}."

    pedido = {
        "origem": origem,
        "destino": destino,
        "acessibilidade": bool(bruto.get("acessibilidade"))
    }
    return pedido, f"✅ Pedido interpretado com sucesso via {fonte}!"

def narrar(r):
    if not r["caminho"]:
        return f"Não existe rota disponível entre {r['origem']} e {r['destino']} com os bloqueios atuais."

    texto = f"Embarque em {r['origem']} e siga até {r['destino']}. Serão {r['paradas']} parada(s) no total"
    if r["baldeacoes"] > 0:
        trocas = ", ".join([f"na estação {est} para a {lin}" for est, lin in r["lista_baldeacoes"]])
        texto += f" com {r['baldeacoes']} baldeação(ões): {trocas}."
    else:
        texto += " sem necessidade de trocar de linha."

    texto += f" Tempo estimado: {r['tempo_min']} minutos."
    if r["alertas"]:
        texto += " ⚠️ Atenção: " + "; ".join(r["alertas"])
    return texto




In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

opcoes = ([(f"📍 {local}", ("local", local)) for local in LOCAIS] +
          [(f"🚆 {estacao}", ("estacao", estacao)) for estacao in TODAS_ESTACOES])

txt_pedido = widgets.Textarea(
    placeholder="Ex.: Tô na Catedral da Sé e quero ir pro MASP de cadeira de rodas",
    layout=widgets.Layout(width="100%", height="60px")
)
btn_interpretar = widgets.Button(
    description="✨ Interpretar Frase (IA)",
    button_style="primary",
    icon="magic",
    layout=widgets.Layout(width="220px")
)
dd_origem = widgets.Dropdown(options=opcoes, description="Origem:", layout=widgets.Layout(width="45%"))
dd_destino = widgets.Dropdown(options=opcoes, value=("local", "MASP"), description="Destino:", layout=widgets.Layout(width="45%"))
chk_acess = widgets.Checkbox(description="Precisa de Acessibilidade", value=False)

sel_fechadas = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Fechadas:", rows=4, layout=widgets.Layout(width="30%"))
sel_manut = widgets.SelectMultiple(options=TODAS_ESTACOES, description="Elevador Off:", rows=4, layout=widgets.Layout(width="30%"))
sel_paralisadas = widgets.SelectMultiple(options=list(LINHAS.keys()), description="Linha Parada:", rows=3, layout=widgets.Layout(width="30%"))

rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Algoritmo:")
btn_buscar = widgets.Button(
    description="🔎 Buscar Rota",
    button_style="success",
    icon="search",
    layout=widgets.Layout(width="220px")
)

saida = widgets.Output()


def ao_interpretar(_):
    with saida:
        clear_output()
        if not txt_pedido.value.strip():
            print("⚠️ Digite uma frase antes de clicar em interpretar.")
            return

        print("🤖 Processando frase...")
        pedido, msg = interpretar_pedido(txt_pedido.value)
        print(msg)

        if pedido:
            dd_origem.value = pedido["origem"]
            dd_destino.value = pedido["destino"]
            chk_acess.value = pedido["acessibilidade"]
            print(f"👉 Campos atualizados no painel: Origem = {pedido['origem'][1]} | Destino = {pedido['destino'][1]} | Acessibilidade = {pedido['acessibilidade']}")

def desenhar_caminho_estilizado(r):
    if not r["caminho"]:
        return """
        <div style='padding:14px; background:#fef2f2; border-left:5px solid #ef4444; color:#991b1b; border-radius:8px; font-weight:bold; margin-top:10px;'>
            🚫 Nenhuma rota disponível com os bloqueios atuais.
        </div>
        """

    caminho = r["caminho"]

    linha_atual = None
    if len(caminho) > 1:
        trecho_inicial = LINHAS_DO_TRECHO.get((caminho[0], caminho[1]), set())
        linha_atual = list(trecho_inicial)[0] if trecho_inicial else "Linha 1-Azul"

    html = """
    <div style="background-color: #0f172a; color: #f8fafc; padding: 20px; border-radius: 12px; font-family: sans-serif; margin-top: 15px; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.3);">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 16px; border-bottom: 1px solid #334155; padding-bottom: 10px; color: #38bdf8; display: flex; align-items: center; gap: 8px;">
            🗺️ <span>Itinerário Detalhado do Percurso</span>
        </div>
        <div style="display: flex; flex-direction: column; gap: 4px;">
    """

    for i in range(len(caminho)):
        est = caminho[i]

        troca = next((b[1] for b in r["lista_baldeacoes"] if b[0] == est), None)
        if troca:
            linha_atual = troca

        if not troca and i < len(caminho) - 1:
            linhas_trecho = LINHAS_DO_TRECHO.get((caminho[i], caminho[i+1]), set())
            if linha_atual not in linhas_trecho and linhas_trecho:
                linha_atual = list(linhas_trecho)[0]

        cor_linha = CORES.get(linha_atual, "#64748b")
        is_origem = (i == 0)
        is_destino = (i == len(caminho) - 1)

        if is_origem:
            tag_status = f"<span style='background:#22c55e; color:#052e16; font-size:11px; font-weight:800; padding:2px 8px; border-radius:12px; margin-right:8px;'>EMBARQUE</span>"
        elif is_destino:
            tag_status = f"<span style='background:#ef4444; color:#ffffff; font-size:11px; font-weight:800; padding:2px 8px; border-radius:12px; margin-right:8px;'>DESEMBARQUE</span>"
        else:
            tag_status = ""

        badge_troca = ""
        if troca:
            badge_troca = f"""
            <div style="margin: 6px 0 6px 28px;">
                <span style="background: #f59e0b; color: #000000; font-size: 12px; font-weight: bold; padding: 4px 12px; border-radius: 20px; display: inline-block; box-shadow: 0 2px 4px rgba(0,0,0,0.3);">
                    🔄 BALDEAÇÃO: Mude para a <span style="text-decoration: underline;">{troca}</span>
                </span>
            </div>
            """

        tag_linha = f"<span style='background:{cor_linha}; color:#ffffff; font-size:10px; font-weight:bold; padding:2px 8px; border-radius:4px; margin-left:8px;'>{linha_atual}</span>"

        html += f"""
        <div style="position: relative;">
            <div style="display: flex; align-items: center; padding: 6px 0;">
                <div style="width: 14px; height: 14px; border-radius: 50%; background-color: {cor_linha}; border: 3px solid #0f172a; flex-shrink: 0; box-shadow: 0 0 0 2px {cor_linha};"></div>
                <div style="margin-left: 12px; font-size: 14px; color: #ffffff; font-weight: {'bold' if (is_origem or is_destino or troca) else 'normal'};">
                    {tag_status}
                    <span style="color: #ffffff;">{est}</span>
                    {tag_linha}
                </div>
            </div>
            {badge_troca}
        </div>
        """

    html += "</div></div>"
    return html

def ao_buscar(_):
    with saida:
        clear_output()
        pedido = {
            "origem": dd_origem.value,
            "destino": dd_destino.value,
            "acessibilidade": chk_acess.value
        }
        r = planejar(
            pedido,
            fechadas=sel_fechadas.value,
            manutencao=sel_manut.value,
            algoritmo=rb_algoritmo.value,
            linhas_paralisadas=sel_paralisadas.value
        )

        display(HTML(f"<h3>🚇 Rota Planejada ({r['algoritmo']}): {r['origem']} → {r['destino']}</h3>"))

        print("🗣️ Instruções do Narrador:")
        print("  ", narrar(r))
        print("\n📊 Dados da Rota:")
        print(f"  • Paradas: {r['paradas']}")
        print(f"  • Baldeações: {r['baldeacoes']} {r['lista_baldeacoes']}")
        print(f"  • Estações Visitadas na busca: {len(r['visitados'])}")
        print(f"  • Regras Lógicas Usadas: {', '.join(r['regras_usadas'])}")

        display(HTML(desenhar_caminho_estilizado(r)))

btn_interpretar.on_click(ao_interpretar)
btn_buscar.on_click(ao_buscar)

painel = widgets.VBox([
    widgets.HTML("<h2>MetrôBot SP 2.0 — Busca por Texto & IA</h2>"),
    widgets.HTML("<b>1. Digite seu pedido em linguagem natural:</b>"),
    txt_pedido,
    btn_interpretar,
    widgets.HTML("<hr><b>2. Ou configure os parâmetros manualmente:</b>"),
    widgets.HBox([dd_origem, dd_destino]),
    widgets.HBox([chk_acess, rb_algoritmo]),
    widgets.HTML("<b>3. Simulação de Bloqueios e Avarias:</b>"),
    widgets.HBox([sel_fechadas, sel_manut, sel_paralisadas]),
    widgets.HTML("<br>"),
    btn_buscar,
    saida
])


In [ ]:

display(painel)